# Example 07 · Multi-Agent: Router

**Source:** [LangChain Docs — Multi-Agent Router](https://docs.langchain.com/oss/python/langchain/multi-agent/router)

---

## Objectives

By the end of this notebook you will be able to:

1. Explain the **Router pattern** and how it differs from the Supervisor pattern
2. Build a **single-agent router** that classifies queries and routes to one specialist using `Command(goto=...)`
3. Build a **parallel multi-agent router** that fans out to multiple agents simultaneously using `Send`
4. Use `Annotated[list, operator.add]` as a reducer to collect results from parallel branches
5. Decide when to choose a Router vs. a Supervisor architecture

---

## Background

### The Router Pattern

A **Router** is a preprocessing step that classifies an incoming query and directs it to the right agent(s).
Unlike a Supervisor — which uses an LLM to *dynamically* orchestrate agents turn-by-turn — a Router makes
a **single classification decision**, then steps aside.

```
User query
    │
    ▼
┌─────────┐  classify  ┌──────────────────┐
│ Router  │ ─────────► │ billing_agent    │  (Single routing)
│  node   │            └──────────────────┘
└─────────┘
    │         fan-out  ┌──────────────────┐
    └────────────────► │ docs_agent       │
                       │ history_agent    │  (Parallel routing)
                       │ usecases_agent   │
                       └──────────────────┘
```

### Two Routing Patterns

| Pattern | API | When to use |
|---------|-----|-------------|
| **Single-agent routing** | `Command(goto="node_name")` | Query belongs to exactly one domain |
| **Parallel fan-out** | `Send("node", state)` list | Query benefits from multiple perspectives simultaneously |

### Router vs. Supervisor

| Dimension | Router | Supervisor |
|-----------|--------|------------|
| Routing decision | One preprocessing step | Dynamic per-turn LLM decision |
| Parallelism | Native via `Send` | Sequential tool calls (default) |
| Memory | Stateless by default | Conversational, stateful |
| Latency | Lower (1 classify call) | Higher (LLM decides each step) |
| Best for | Distinct domains, parallel research | Multi-step tasks, unknown depth |

Run cells **top to bottom**.

## Setup

In [ ]:
import sys, operator
from pathlib import Path
from typing import TypedDict, Annotated, Literal

# 添加项目根目录，使 common/ 包可被导入
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent

from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# 初始化 Langfuse 追踪处理器（可选）
_handler = create_langfuse_handler()

def make_config(session_id: str = 's07', trace_name: str | None = None) -> dict:
    return build_langfuse_config(_handler, session_id=session_id, trace_name=trace_name or session_id)

print("✓ Setup complete")

---

## Part 1 · Single-Agent Routing with `Command`

The simplest router pattern: one LLM call classifies the query, then
`Command(goto="node_name")` sends execution directly to the matching specialist.

```
START → router_node ──► billing_agent   → END
                    ──► technical_agent → END
                    ──► general_agent   → END
```

The graph compiles all three destination nodes, but only **one** executes per query.

### Step 1a — Define specialist tools and agent nodes

In [ ]:
# ── Mock tools for each specialist domain ────────────────────────────────────

@tool
def lookup_invoice(invoice_id: str) -> str:
    # 模拟账单数据库查询
    db = {
        'INV-001': 'Paid · $299.00 · 2024-01-15',
        'INV-002': 'Overdue · $150.00 · due 2024-02-01',
        'INV-003': 'Pending · $89.00 · due 2024-03-10',
    }
    return db.get(invoice_id.upper(), f'No record for {invoice_id}')

@tool
def check_service_status(service: str) -> str:
    # 模拟服务健康状态检查
    statuses = {
        'api':      '✓ Operational — avg latency 42ms',
        'database': '✓ Operational — 118/500 connections',
        'auth':     '⚠ Degraded — retries every 5s',
        'cdn':      '✓ Operational — cache hit 94%',
    }
    return statuses.get(service.lower(), f'Unknown service: {service}')

print("✓ Tools defined")

# ── Specialist agent nodes ────────────────────────────────────────────────────

def billing_agent_node(state: dict) -> dict:
    # 账单专家智能体：只配备账单查询工具
    agent = create_agent(
        model=create_llm(),
        tools=[lookup_invoice],
        system_prompt=(
            'You are a billing support specialist. '
            'Help customers with invoice lookups and payment status. '
            'Be concise and friendly.'
        ),
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-billing'),
    )
    return {'answer': f"[Billing] {result['messages'][-1].content}"}

def technical_agent_node(state: dict) -> dict:
    # 技术支持智能体：只配备服务状态检查工具
    agent = create_agent(
        model=create_llm(),
        tools=[check_service_status],
        system_prompt=(
            'You are a technical support engineer. '
            'Help customers diagnose system issues and check service health. '
            'Be precise and technical.'
        ),
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-technical'),
    )
    return {'answer': f"[Technical] {result['messages'][-1].content}"}

def general_agent_node(state: dict) -> dict:
    # 通用客服智能体：无工具，直接用 LLM 知识回答
    agent = create_agent(
        model=create_llm(),
        tools=[],
        system_prompt='You are a helpful customer support agent. Answer concisely.',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-general'),
    )
    return {'answer': f"[General] {result['messages'][-1].content}"}

print("✓ Agent nodes defined")

### Step 1b — Build the router node

In [ ]:
# ── Graph state ───────────────────────────────────────────────────────────────
class SupportState(TypedDict):
    query:  str   # 传入的用户查询
    answer: str   # 专业智能体的最终回答

# ── Query classifier ──────────────────────────────────────────────────────────
def classify_query(query: str) -> str:
    # 用最小化的 LLM 调用对查询分类，避免消耗多余 token
    llm = create_llm()
    response = llm.invoke([
        HumanMessage(content=(
            'Classify this support query into EXACTLY ONE: billing, technical, general.\n'
            f'Query: {query}\n'
            'Reply with ONE lowercase word only.'
        ))
    ])
    cat = response.content.strip().lower().split()[0]
    # 防御性处理：分类结果不合法时默认 general
    return cat if cat in {'billing', 'technical', 'general'} else 'general'

# ── Router node ───────────────────────────────────────────────────────────────
def router_node(state: SupportState) -> Command[Literal['billing_agent', 'technical_agent', 'general_agent']]:
    # 分类查询，通过 Command(goto=...) 将执行转移到对应的专业智能体节点
    category = classify_query(state['query'])
    print(f'  → Classified as: [{category}]')
    # Command(goto=...) 指定下一步要执行的节点名称
    return Command(goto=f'{category}_agent')

print("✓ Router node defined")

### Step 1c — Assemble and compile the graph

In [ ]:
# ── Build StateGraph ──────────────────────────────────────────────────────────
builder = StateGraph(SupportState)

# 注册所有节点
builder.add_node('router',           router_node)
builder.add_node('billing_agent',    billing_agent_node)
builder.add_node('technical_agent',  technical_agent_node)
builder.add_node('general_agent',    general_agent_node)

# 图入口：所有查询先经过路由节点
builder.add_edge(START, 'router')

# 各专业智能体完成后直接结束
# 路由节点通过 Command(goto=...) 动态决定路径，无需显式条件边
builder.add_edge('billing_agent',   END)
builder.add_edge('technical_agent', END)
builder.add_edge('general_agent',   END)

# 编译图（无状态路由器不需要 checkpointer）
support_router = builder.compile()
print("✓ Single-agent router compiled")
print('Nodes:', list(support_router.graph.nodes))

### Step 1d — Run the router

In [ ]:
# 用三类不同查询测试路由器
test_queries = [
    'Can you check the status of invoice INV-002?',
    'The auth service seems slow — can you check its health?',
    'What are your support hours on weekends?',
]

for query in test_queries:
    print('=' * 60)
    print(f'Query: {query}')
    result = support_router.invoke(
        {'query': query, 'answer': ''},
        config=make_config('s07-part1'),
    )
    print(f"Answer: {result['answer']}")
    print()

---

## Part 2 · Parallel Routing with `Send`

Sometimes a query benefits from multiple perspectives at once. The **fan-out** pattern
dispatches to several agents *simultaneously* using `Send`, then collects all results
via an `operator.add` reducer on a shared list field.

```
                   ┌──► docs_agent    ─┐
START → fan_out ──►│    history_agent ─┼──► (sub_results merged) → END
                   └──► usecases_agent─┘
```

**Key:** `sub_results: Annotated[list[str], operator.add]`
Each parallel agent appends one string; `operator.add` merges all three lists automatically.

### Step 2a — Define the parallel state and agent nodes

In [ ]:
# ── Mock research tools ───────────────────────────────────────────────────────

@tool
def search_docs(keyword: str) -> str:
    # 模拟技术文档搜索
    docs = {
        'python':     'Python is a high-level interpreted language emphasizing readability.',
        'javascript': 'JavaScript is an event-driven single-threaded language for the web.',
        'rust':       'Rust guarantees memory safety through its ownership system at compile time.',
        'go':         'Go is statically typed, designed for simplicity and concurrency.',
    }
    for k, v in docs.items():
        if k in keyword.lower(): return v
    return f'No documentation found for {keyword!r}'

@tool
def search_history(topic: str) -> str:
    # 模拟技术历史数据库
    history = {
        'python':     'Created by Guido van Rossum, first released in 1991.',
        'javascript': 'Designed by Brendan Eich in 10 days in 1995 at Netscape.',
        'rust':       'Originated at Mozilla in 2006; 1.0 released in 2015.',
        'go':         'Designed at Google by Pike, Thompson, Griesemer; released 2009.',
    }
    for k, v in history.items():
        if k in topic.lower(): return v
    return f'No history found for {topic!r}'

@tool
def get_use_cases(technology: str) -> str:
    # 模拟应用场景数据库
    cases = {
        'python':     'Data science, ML/AI, web backends (Django/FastAPI), scripting.',
        'javascript': 'Frontend UIs, Node.js backends, mobile (React Native), full-stack.',
        'rust':       'Systems programming, WebAssembly, embedded, high-perf networking.',
        'go':         'Cloud infra, CLI tools, microservices, DevOps tooling.',
    }
    for k, v in cases.items():
        if k in technology.lower(): return v
    return f'No use cases found for {technology!r}'

print("✓ Research tools defined")

# ── Parallel state (operator.add reducer on sub_results) ──────────────────────
class ResearchState(TypedDict):
    query: str
    # Annotated + operator.add: 每个并行分支追加一个字符串，最终合并为完整列表
    sub_results: Annotated[list[str], operator.add]

# ── Parallel agent nodes ──────────────────────────────────────────────────────
# 每个节点通过 Send 接收独立状态副本；返回 sub_results 列表追加到父图状态

def docs_agent_node(state: dict) -> dict:
    # 技术文档智能体：解释该技术「是什么」
    agent = create_agent(
        model=create_llm(), tools=[search_docs],
        system_prompt='You are a technical documentation expert. Provide accurate, concise descriptions.',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-docs'),
    )
    return {'sub_results': [f"[Docs]    {result['messages'][-1].content}"]}

def history_agent_node(state: dict) -> dict:
    # 历史背景智能体：解释该技术「从哪里来」
    agent = create_agent(
        model=create_llm(), tools=[search_history],
        system_prompt='You are a technology historian. Provide historical context and origin stories.',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-history'),
    )
    return {'sub_results': [f"[History] {result['messages'][-1].content}"]}

def usecases_agent_node(state: dict) -> dict:
    # 应用场景智能体：解释该技术「用在哪里」
    agent = create_agent(
        model=create_llm(), tools=[get_use_cases],
        system_prompt='You are a technology advisor. Explain practical real-world applications.',
    )
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': state['query']}]},
        config=make_config('s07-usecases'),
    )
    return {'sub_results': [f"[Cases]   {result['messages'][-1].content}"]}

print("✓ Parallel agent nodes defined")

### Step 2b — Fan-out router using `Send`

In [ ]:
# ── Fan-out router (conditional edge function, NOT a node) ────────────────────
def fan_out_router(state: ResearchState) -> list:
    # 返回 Send 对象列表：每个 Send 为一个并行分支创建独立的状态副本
    print(f"  → Fan-out: dispatching to docs / history / use-cases agents in parallel")
    return [
        Send('docs_agent',     {'query': state['query'], 'sub_results': []}),
        Send('history_agent',  {'query': state['query'], 'sub_results': []}),
        Send('usecases_agent', {'query': state['query'], 'sub_results': []}),
    ]

# ── Build parallel graph ───────────────────────────────────────────────────────
builder2 = StateGraph(ResearchState)
builder2.add_node('docs_agent',     docs_agent_node)
builder2.add_node('history_agent',  history_agent_node)
builder2.add_node('usecases_agent', usecases_agent_node)

# 从 START 通过 fan_out_router 条件边扇出到三个并行节点
# 第三个参数列出所有可能的目标节点（供 LangGraph 静态图分析使用）
builder2.add_conditional_edges(
    START, fan_out_router,
    ['docs_agent', 'history_agent', 'usecases_agent'],
)
# 各并行节点完成后直接进入 END；operator.add 已将各分支结果合并到主图状态
builder2.add_edge('docs_agent',     END)
builder2.add_edge('history_agent',  END)
builder2.add_edge('usecases_agent', END)

research_router = builder2.compile()
print("✓ Parallel fan-out router compiled")

### Step 2c — Run the parallel router

In [ ]:
query = 'Tell me about the Python programming language'
print(f'Query: {query}')
print('=' * 60)

result = research_router.invoke(
    {'query': query, 'sub_results': []},
    config=make_config('s07-part2'),
)

print('\n── Results from all parallel agents ──────────────────────')
for entry in result['sub_results']:
    print(f'\n{entry}')
print(f"\n✓ {len(result['sub_results'])} agents responded")

---

## Part 3 · Router vs. Supervisor — When to Use Which

### Decision Guide

```
Is routing logic known upfront (rule-based or single LLM classify)?
    YES  → Router
    NO   → Supervisor

Do multiple agents need to answer simultaneously?
    YES  → Router (parallel fan-out with Send)
    NO   → Router (single routing with Command) or Supervisor

Does the orchestrator need memory of prior tool calls?
    YES  → Supervisor
    NO   → Router

Is the number of agent calls per query variable and unknown?
    YES  → Supervisor
    NO   → Router
```

### Side-by-Side

| Dimension | Router | Supervisor |
|-----------|--------|------------|
| **Routing decision** | Preprocessing (1 LLM call or rule) | Dynamic, per-turn LLM decision |
| **Parallelism** | Native with `Send` | Sequential tool calls (default) |
| **Memory** | Stateless by default | Conversational, stateful |
| **Agents per query** | Fixed: 1 or N | Variable: decided at runtime |
| **Best for** | Distinct domains, parallel research | Multi-step tasks, unknown depth |
| **LangGraph API** | `Command(goto=...)`, `Send(...)` | `create_agent` with subagent tools |
| **Latency** | Lower (no per-step LLM overhead) | Higher (LLM decides each step) |

---

## Summary

| Concept | API | Notes |
|---------|-----|-------|
| Graph state | `TypedDict` | Typed state shared across all nodes |
| State reducer | `Annotated[list, operator.add]` | Merges parallel branch outputs automatically |
| Single routing | `Command(goto="node_name")` | Returned from a router *node* |
| Parallel routing | `[Send("node", state), ...]` | Returned from a *conditional edge* function |
| Graph entry (normal) | `add_edge(START, "router")` | Route through a node first |
| Graph entry (fan-out) | `add_conditional_edges(START, fn, [...])` | Fan-out directly from START |

### Key Takeaways

1. **Router = classify once, step aside** — minimal LLM overhead (one classification call)
2. **`Command(goto=...)` is for single routing** — returned from a *node*, not a conditional edge
3. **`Send` is for parallel fan-out** — each `Send` spawns an independent branch with its own state copy
4. **`operator.add` is the glue** — silently merges every parallel branch's output into one list
5. **Routers are stateless by default** — add `InMemorySaver` only if multi-turn memory is needed

In [ ]:
print(f'Traces available at: {get_langfuse_host()}')